# Fine-tune `vietnamese-bi-encoder` cho bài toán "cùng sự kiện"

Notebook chạy trên **Kaggle** (GPU **T4 x1**, bật **Internet**). Dữ liệu: Kaggle Dataset tạo từ thư mục
`data/temp/kaggle_upload/` (sinh bằng `training/prepare_kaggle_data.py`).

**Luồng:** cấu hình → nạp dữ liệu → chia train/dev theo thời gian, loại cửa sổ đánh giá → đo zero-shot → huấn luyện
→ đo lại, so ghép cặp với zero-shot → lưu mô hình và `results_<data>_<loss>.json` vào `/kaggle/working`.

**Tiền xử lý bắt buộc** (mọi nơi dùng mô hình, kể cả sau này trong gom cụm/API): `NFC` → tách từ `pyvi` →
`model.encode(..., normalize_embeddings=True)`. Đầu vào là `tiêu đề + ". " + sapo`.

**Đánh giá** theo giao thức của nhóm: chọn ngưỡng cosine trên **tập chỉnh** (`audit_v1`), đo trên **tập đo**
(`audit_v2`, holdout) với precision/recall **có trọng số theo tầng**, CI 95% bootstrap, recall theo tầng trùng
từ vựng. Thêm hai phép đo:
- **So ghép cặp** với zero-shot: bootstrap trên *cùng* các cặp → CI của phần chênh F1/AP. Hai CI riêng chồng nhau
  không có nghĩa là không khác; CI của phần chênh không chứa 0 mới là bằng chứng.
- **Bộ dò "cùng câu chuyện, khác sự kiện"** (`probe_story.csv`, 137 cặp holdout lấy bằng luật): tỉ lệ cặp có cosine
  ≥ ngưỡng = tỉ lệ sẽ bị **gộp nhầm** khi gom cụm. Fine-tune làm tỉ lệ này tăng → mô hình đang học "cùng chủ đề".

Nhãn đối chiếu là của **Claude, chưa phải người**.

## 0. Cấu hình

- `data`: `"v4"` (mặc định — cặp có **4 mức liên quan**: 1,0 cùng sự kiện · 0,6 cùng câu chuyện · 0,3 cùng chủ đề ·
  0 không liên quan — chỉ dùng với `cosent`/`angle`), `"v3"` (bộ ba nhãn yếu v3, precision ≈ 0,93) hoặc
  `"temp"` (file tạm trên R2, nhãn kiểu "cùng chủ đề", chỉ để đối chứng).
- `loss`: `"gist"` (mặc định — lọc negative giả trong batch bằng mô hình zero-shot), `"mnrl"` (mốc so sánh),
  `"cosent"` / `"angle"` (dùng cặp có điểm: `temp` → nhãn 0/1, `v3` → điểm mềm p, `v4` → 4 mức).
- `use_negatives`: với `v3` + `gist`/`mnrl`, thêm cột negative khó của bộ ba.
- `group_batches`: mỗi **cụm sự kiện** (thành phần liên thông của cặp dương) xuất hiện tối đa một lần trong một
  batch → hai bài cùng sự kiện không bị coi là negative của nhau. `max_pairs_per_cluster` giới hạn số cặp dương
  mỗi cụm để vài cụm lớn không áp đảo.
- `seeds`: `[42, 43, 44]` (quy tắc ≥ 3 seed). `epochs=1`: lần chạy 10/10 cho thấy MRR dev đứng yên từ epoch 0,9.
- `smoke=True`: chạy thử vài bước để kiểm tra notebook (không dùng kết quả).
- Ghi đè nhanh bằng biến môi trường: `DATA`, `LOSS`, `MODEL_NAME`, `SMOKE`.

In [ ]:
import os

CFG = dict(
    model_name="bkai-foundation-models/vietnamese-bi-encoder",
    data="v4",                   # "v4" (mặc định, 4 mức) | "v3" | "temp"
    loss="cosent",               # "cosent" | "angle" (cho v4) · "gist" | "mnrl" (cho v3/temp)
    use_negatives=True,          # v3 + gist/mnrl: cột negative khó
    group_batches=True,          # mỗi cụm sự kiện tối đa 1 lần / batch
    max_pairs_per_cluster=20,    # giới hạn cặp dương mỗi cụm — CHỈ áp cho gist/mnrl (chống negative giả trong batch)
    seeds=[42, 43, 44],
    lr=2e-5, epochs=1, batch_size=64, warmup_ratio=0.1, weight_decay=0.01,
    temperature=0.05,            # bkai được huấn luyện với MNRL scale 20 = temperature 0,05
    max_seq_length=256,          # giới hạn cứng của PhoBERT; tiêu đề + sapo sau tách từ p99 ≈ 92 token
    holdout_start="2026-09-13T00:00:00+07:00",  # khớp nhãn yếu v3 → tập đo audit_v2 nằm trọn sau mốc này
    gap_hours=48,                # bỏ thêm 48h trước holdout khỏi dữ liệu train
    dev_frac=0.10,               # 10% cặp dương muộn nhất (trước khoảng đệm) làm dev
    eval_steps=50,
    n_boot=1000,
    smoke=False,
)
# Chạy ngoài Kaggle (kiểm tra local) hoặc đổi nhanh cấu hình: ghi đè qua biến môi trường
for key, env in [("model_name", "MODEL_NAME"), ("data", "DATA"), ("loss", "LOSS")]:
    CFG[key] = os.environ.get(env, CFG[key])
CFG["smoke"] = os.environ.get("SMOKE", str(CFG["smoke"])).lower() in ("1", "true")
if CFG["smoke"]:
    CFG.update(seeds=CFG["seeds"][:1], epochs=1, eval_steps=2, n_boot=50)
assert CFG["data"] in ("v3", "v4", "temp") and CFG["loss"] in ("gist", "mnrl", "cosent", "angle")
assert CFG["data"] != "v4" or CFG["loss"] in ("cosent", "angle"), "v4 là cặp có điểm 4 mức → cosent hoặc angle"
CFG

## 1. Môi trường

In [ ]:
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")  # 1 GPU: T4 x2 sẽ chạy DataParallel, chia đôi batch → mất negative trong batch
os.environ["WANDB_DISABLED"] = "true"               # ảnh Kaggle có sẵn wandb, Trainer sẽ dừng chờ API key
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import importlib.util, subprocess, sys
from importlib.metadata import version

def _need(pkg, mod, min_major_minor=None):
    if importlib.util.find_spec(mod) is None:
        return True
    if min_major_minor:
        major, minor = (int(x) for x in version(pkg).split(".")[:2])
        return (major, minor) < min_major_minor
    return False

missing = [spec for spec, pkg, mod, mv in [
    ("pyvi", "pyvi", "pyvi", None),
    ("sentence-transformers>=4.0", "sentence-transformers", "sentence_transformers", (4, 0)),
    ("datasets", "datasets", "datasets", None),
] if _need(pkg, mod, mv)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="sentence_transformers")

import torch, sentence_transformers, transformers
print("torch", torch.__version__, "| sentence-transformers", sentence_transformers.__version__,
      "| transformers", transformers.__version__, "| pyvi", version("pyvi"))
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không có (CPU)")

## 2. Nạp dữ liệu

In [ ]:
import csv, json, random, unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
from pyvi import ViTokenizer

NEED = ["articles.jsonl", "v3_triplets.csv", "v3_pairs.csv", "weak_supervision_pairs.csv", "audit_v1.csv",
        "audit_v2.csv", "strata_sizes.json", "probe_story.csv"] + (["v4_pairs.csv"] if CFG["data"] == "v4" else [])
_kaggle = Path("/kaggle/input")
if _kaggle.exists():
    # os.walk(followlinks=True): thư mục input của Kaggle có thể là symlink, Path.rglob không đi theo
    _dirs = {Path(d): set(fs) for d, _, fs in os.walk(_kaggle, followlinks=True) if "articles.jsonl" in fs}
    _ok = [d for d, fs in _dirs.items() if set(NEED) <= fs]
    if not _ok:
        raise FileNotFoundError(
            "Không tìm thấy thư mục dữ liệu đủ file trong /kaggle/input. Kiểm tra: (1) đã Add Input dataset chưa, "
            "(2) dataset đã là PHIÊN BẢN MỚI (có v3_triplets.csv, probe_story.csv…) chưa — trong panel Input bấm ⋮ "
            "→ chọn version mới nhất. Đang thấy: "
            + ("; ".join(f"{d}: thiếu {sorted(set(NEED) - fs)}" for d, fs in _dirs.items())
               or str(sorted(str(x) for x in _kaggle.iterdir()))))
    DATA = _ok[0]
else:
    DATA = Path(os.environ.get("DATA_DIR", "data/temp/kaggle_upload"))
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("OUT_DIR", "out"))
OUT.mkdir(parents=True, exist_ok=True)
print("DATA =", DATA, "| OUT =", OUT)

arts = {}
with open(DATA / "articles.jsonl", encoding="utf-8") as f:
    for line in f:
        a = json.loads(line)
        a["t"] = datetime.fromisoformat(a["published_at"])
        arts[a["article_id"]] = a

def prep(text: str) -> str:
    # Tiền xử lý DUY NHẤT — dùng y hệt khi train, khi đánh giá, và khi encode thật
    return ViTokenizer.tokenize(unicodedata.normalize("NFC", text))

_seg = {}
def seg(aid: str) -> str:
    if aid not in _seg:
        _seg[aid] = prep(arts[aid]["text"])
    return _seg[aid]

def read_csv(name):
    with open(DATA / name, encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

# Mọi nguồn quy về một dạng: a, b (cặp), neg (negative khó hoặc None), label 0/1, score (cho CoSENT)
if CFG["data"] == "temp":
    rows = [dict(a=r["article_1_id"], b=r["article_2_id"], neg=None, label=int(r["label"]),
                 score=float(r["label"])) for r in read_csv("weak_supervision_pairs.csv")]
elif CFG["data"] == "v4":
    # label = 1 chỉ cho mức "cùng sự kiện" (dùng để gom cụm, chọn dev); score = 1,0 / 0,6 / 0,3 / 0
    rows = [dict(a=r["a"], b=r["b"], neg=None, label=int(r["level"] == "3"), score=float(r["score"]))
            for r in read_csv("v4_pairs.csv")]
elif CFG["loss"] in ("cosent", "angle"):
    rows = [dict(a=r["a"], b=r["b"], neg=None, label=int(r["label"]), score=float(r["score"]))
            for r in read_csv("v3_pairs.csv")]
else:
    rows = [dict(a=r["anchor_id"], b=r["positive_id"], neg=(r["negative_id"] or None) if CFG["use_negatives"] else None,
                 label=1, score=float(r["p"])) for r in read_csv("v3_triplets.csv")]
audit_v1 = read_csv("audit_v1.csv")
audit_v2 = read_csv("audit_v2.csv")
probe = read_csv("probe_story.csv")
v4_test = read_csv("v4_test.csv") if (DATA / "v4_test.csv").exists() else []   # tập đo 4 mức (holdout), tuỳ chọn
v4_hard = read_csv("v4_test_hard.csv") if (DATA / "v4_test_hard.csv").exists() else []   # cặp cùng nội dung, khác chữ
strata_sizes = json.loads((DATA / "strata_sizes.json").read_text(encoding="utf-8"))

print(f"[{CFG['data']}/{CFG['loss']}] {len(rows)} dòng · nhãn {dict(Counter(r['label'] for r in rows))} · "
      f"điểm {dict(sorted(Counter(r['score'] for r in rows).items()))} · "
      f"có negative khó {sum(r['neg'] is not None for r in rows)} · {len(arts)} bài")
print(f"tập chỉnh {len(audit_v1)} cặp · tập đo {len(audit_v2)} cặp · tổng thể {sum(strata_sizes.values())} cặp · "
      f"bộ dò {len(probe)} cặp {dict(Counter(r['kind'] for r in probe))}")

## 3. Chia dữ liệu — chống rò rỉ, gom cụm

1. **Bỏ mọi dòng có bài (kể cả negative) đăng từ (holdout − 48h) trở đi** → train không chạm cửa sổ của tập đo.
2. Bỏ thêm dòng có bài nằm trong tập đo hoặc trùng nguyên cặp với tập chỉnh (phòng hờ).
3. **Dev** = 10% cặp dương (khác nhau) muộn nhất còn lại; dòng train có bài thuộc dev bị bỏ.
4. **Cụm sự kiện** = thành phần liên thông của các cặp dương train. Mỗi cụm giữ tối đa `max_pairs_per_cluster` cặp
   (chọn ngẫu nhiên, cố định theo seed 0 để mọi seed train trên cùng dữ liệu).

Dev đo bằng **truy hồi**: mỗi bài dev phải tìm ra bài cùng sự kiện giữa **mọi bài cùng khoảng thời gian** (các bài
cùng thời điểm, cùng chủ đề chính là negative khó tự nhiên). Dev chỉ để chọn checkpoint, không báo cáo.

In [ ]:
cut = datetime.fromisoformat(CFG["holdout_start"]) - timedelta(hours=CFG["gap_hours"])
v2_art = {i for r in audit_v2 for i in (r["a"], r["b"])}
v1_keys = {frozenset((r["a"], r["b"])) for r in audit_v1}
row_ids = lambda r: [r["a"], r["b"]] + ([r["neg"]] if r["neg"] else [])

usable = [r for r in rows if all(arts[i]["t"] < cut for i in row_ids(r))
          and not (set(row_ids(r)) & v2_art) and frozenset((r["a"], r["b"])) not in v1_keys]

_pos = {}
for r in usable:
    if r["label"] == 1:
        _pos.setdefault(frozenset((r["a"], r["b"])), r)
pos = sorted(_pos.values(), key=lambda r: max(arts[r["a"]]["t"], arts[r["b"]]["t"]))
k = int(len(pos) * (1 - CFG["dev_frac"]))
dev_pos = pos[k:]
dev_art = {i for p in dev_pos for i in (p["a"], p["b"])}
dev_from = min(arts[i]["t"] for i in dev_art)
train_rows = [r for r in usable if not (set(row_ids(r)) & dev_art)]

# Cụm sự kiện: union-find trên cặp dương train
_parent = {}
def find(x):
    _parent.setdefault(x, x)
    while _parent[x] != x:
        _parent[x] = _parent[_parent[x]]
        x = _parent[x]
    return x
for r in train_rows:
    if r["label"] == 1:
        _parent[find(r["a"])] = find(r["b"])
cluster = lambda aid: find(aid) if aid in _parent else aid   # bài không có cặp dương: cụm riêng
cl_size = Counter(cluster(i) for i in _parent)

by_cl = defaultdict(list)
for r in train_rows:
    if r["label"] == 1:
        by_cl[cluster(r["a"])].append(r)
_rng = random.Random(0)
capped = []
# CoSENT/AnglE so trực tiếp các cặp có nhãn, không dùng negative trong batch → không cần giới hạn (v4: giữ 6.406
# cặp mức 3 thay vì 4.672). gist/mnrl: cụm lớn sinh negative giả trong batch → giới hạn.
cap = None if CFG["loss"] in ("cosent", "angle") else CFG["max_pairs_per_cluster"]
for c, rs in by_cl.items():
    # v3: nhiều dòng chung một cặp (khác negative) → giới hạn theo CẶP, giữ mọi negative của cặp được chọn
    keys = sorted({frozenset((r["a"], r["b"])) for r in rs}, key=sorted)
    keep = set(_rng.sample(keys, cap)) if cap and len(keys) > cap else set(keys)
    capped += [r for r in rs if frozenset((r["a"], r["b"])) in keep]
n_before = sum(r["label"] == 1 for r in train_rows)
train_rows = capped + [r for r in train_rows if r["label"] == 0]

# Bể truy hồi dev: mọi bài trong khoảng thời gian dev (kể cả bài không có cặp dương nào)
dev_pool = sorted(i for i, a in arts.items() if dev_from <= a["t"] < cut and i not in v2_art)
if CFG["smoke"]:
    dev_pool = sorted(dev_art | set(random.Random(0).sample([i for i in dev_pool if i not in dev_art], 500)))

print(f"loại vì chạm cửa sổ đánh giá / tập đo / tập chỉnh: {len(rows) - len(usable)} dòng")
print(f"cụm sự kiện train: {len(cl_size)} · lớn nhất {[n for _, n in cl_size.most_common(3)]} bài · "
      f"cặp dương {n_before} → {len(capped)}" + (f" sau giới hạn {cap}/cụm" if cap else " (không giới hạn)"))
print(f"train: {len(train_rows)} dòng ({dict(Counter(r['label'] for r in train_rows))})")
print(f"dev  : {len(dev_pos)} cặp dương · bể truy hồi {len(dev_pool)} bài · từ {dev_from:%Y-%m-%d %H:%M}")
assert not ({i for r in train_rows for i in row_ids(r)} & (v2_art | dev_art)), "rò rỉ train ↔ dev/tập đo"
assert all(arts[i]["t"] >= datetime.fromisoformat(CFG["holdout_start"]) for r in probe for i in (r["a"], r["b"]))

## 4. Bộ đánh giá: truy hồi trên dev (chọn checkpoint), giao thức audit và bộ dò (báo cáo)

In [ ]:
from sentence_transformers.evaluation import SentenceEvaluator


class PairRetrievalEvaluator(SentenceEvaluator):
    # Với mỗi bài có cặp dương trong dev: xếp hạng mọi bài khác trong bể theo cosine → MRR, Recall@k.
    def __init__(self, pos_pairs, pool, name="dev", batch_size=128):
        super().__init__()
        self.name, self.batch_size, self.primary_metric = name, batch_size, "mrr"
        self.pool = list(pool)
        idx = {a: i for i, a in enumerate(self.pool)}
        self.rel = defaultdict(set)
        for p in pos_pairs:
            if p["a"] in idx and p["b"] in idx:
                self.rel[idx[p["a"]]].add(idx[p["b"]])
                self.rel[idx[p["b"]]].add(idx[p["a"]])
        self.queries = sorted(self.rel)

    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        E = model.encode([seg(a) for a in self.pool], batch_size=self.batch_size,
                         normalize_embeddings=True, convert_to_tensor=True, show_progress_bar=False)
        q = torch.tensor(self.queries, device=E.device)
        S = E[q] @ E.T
        S[torch.arange(len(q)), q] = -2.0                        # bỏ chính nó
        order = torch.argsort(S, dim=1, descending=True).cpu().numpy()
        rr, r1, r10 = [], [], []
        for row, qi in zip(order, self.queries):
            hit = np.isin(row, list(self.rel[qi]))
            first = int(np.argmax(hit)) if hit.any() else len(row)
            rr.append(1.0 / (first + 1))
            r1.append(float(hit[:1].any()))
            r10.append(hit[:10].sum() / len(self.rel[qi]))
        metrics = {"mrr": float(np.mean(rr)), "hit@1": float(np.mean(r1)), "recall@10": float(np.mean(r10))}
        metrics = self.prefix_name_to_metrics(metrics, self.name)
        self.store_metrics_in_model_card_data(model, metrics, epoch, steps)
        return metrics


# --- Dữ liệu đánh giá cố định, tính một lần ---
Y1 = np.array([int(r["label_llm"]) for r in audit_v1])
Y2 = np.array([int(r["label_llm"]) for r in audit_v2])
ST2 = [f"{r['p_bin']}|{r['cross_source']}" for r in audit_v2]
_cnt = Counter(ST2)
# Horvitz–Thompson: mỗi cặp mẫu nặng N_tầng / n_tầng (giống src/eventrank/weak_labels/evaluate.py).
# Bootstrap lấy mẫu lại TRONG tầng nên n_tầng không đổi → trọng số mỗi cặp cố định.
W2 = np.array([strata_sizes.get(s, 0) / _cnt[s] for s in ST2])
STRATUM_IDX = [np.array([i for i, s in enumerate(ST2) if s == st]) for st in sorted(_cnt)]
H2 = np.array([float(r["cos_head"]) for r in audit_v2])
_lo, _hi = np.quantile(H2[Y2 == 1], [1 / 3, 2 / 3])
TIERS = {"low": H2 < _lo, "mid": (H2 >= _lo) & (H2 < _hi), "high": H2 >= _hi}
ALL_EVAL_IDS = sorted({i for r in audit_v1 + audit_v2 + probe + v4_test + v4_hard for i in (r["a"], r["b"])})
L4 = np.array([int(r["level"]) for r in v4_test])
L4H = np.array([int(r["level"]) for r in v4_hard])


def pair_scores(model):
    # cosine của mọi cặp đánh giá; lưu lại trong results để so ghép cặp về sau
    E = dict(zip(ALL_EVAL_IDS, model.encode([seg(i) for i in ALL_EVAL_IDS], batch_size=64,
                                            normalize_embeddings=True, show_progress_bar=False)))
    cos = lambda rs: np.array([float(E[r["a"]] @ E[r["b"]]) for r in rs])
    return dict(v1=cos(audit_v1), v2=cos(audit_v2), probe=cos(probe), v4=cos(v4_test), v4h=cos(v4_hard))


def graded_metrics(s4, L=None):
    # Tập đo 4 mức: cosine trung bình mỗi mức + AUC giữa hai mức liền kề = xác suất một cặp mức cao có cosine
    # lớn hơn một cặp mức thấp (0,5 = không phân biệt; 1 = luôn đúng thứ tự). Chưa có tập v4 → {}.
    L = L4 if L is None else L
    if not len(s4):
        return {}
    out = {f"mean_cos_level{k}": float(s4[L == k].mean()) for k in (3, 2, 1, 0) if (L == k).any()}
    for hi, lo in ((3, 2), (2, 1), (1, 0)):
        a, b = s4[L == hi], s4[L == lo]
        if len(a) and len(b):
            out[f"auc_{hi}v{lo}"] = float(((a[:, None] > b[None, :]).mean() + 0.5 * (a[:, None] == b[None, :]).mean()))
    from scipy.stats import spearmanr
    out["spearman"] = float(spearmanr(s4, L).statistic)
    return out


def hard_metrics(sh, th):
    # Tập "cùng nội dung, khác chữ": ở NGƯỠNG của chính mô hình, bao nhiêu cặp mức 3 được gộp (recall khác chữ —
    # điểm yếu của bản v4 đầu tiên) và bao nhiêu cặp mức ≤ 2 bị gộp nhầm. Chưa có tập này → {}.
    if not len(sh):
        return {}
    pos, neg = L4H == 3, L4H <= 2
    return dict(n=int(len(sh)), n_level3=int(pos.sum()),
                recall_hard=float((sh[pos] >= th).mean()) if pos.any() else float("nan"),
                false_merge_hard=float((sh[neg] >= th).mean()) if neg.any() else float("nan"),
                **{f"hard_{k}": v for k, v in graded_metrics(sh, L4H).items() if k.startswith("auc")})


def best_threshold(s, y):
    # ngưỡng cho F1 cao nhất (dự đoán dương khi s ≥ ngưỡng); hoà thì lấy ngưỡng nhỏ nhất
    o = np.argsort(-s, kind="stable")
    so, yo = s[o], y[o]
    f1 = 2 * np.cumsum(yo) / (np.arange(1, len(s) + 1) + y.sum())
    f1[np.r_[so[1:] == so[:-1], False]] = -1                # chỉ xét vị trí cuối của mỗi giá trị
    return float(so[len(f1) - 1 - np.argmax(f1[::-1])])


def metrics_at(s2, th, idx):
    # P/R/F1 có trọng số + AP có trọng số trên các cặp idx của tập đo
    y, w, d = Y2[idx], W2[idx], s2[idx] >= th
    tp, npos, npred = (w * y * d).sum(), (w * y).sum(), (w * d).sum()
    p = tp / npred if npred else float("nan")
    r = tp / npos if npos else float("nan")
    f = 2 * p * r / (p + r) if p + r else float("nan")
    o = np.argsort(-s2[idx], kind="stable")
    wy = (w * y)[o]
    ap = float(np.sum(np.cumsum(wy) / np.cumsum(w[o]) * wy) / np.sum(wy))
    return p, r, f, ap


def boot_v2(rng):
    return np.concatenate([rng.choice(ix, len(ix)) for ix in STRATUM_IDX])


def summarize(sc, seed=0):
    th = best_threshold(sc["v1"], Y1)                       # 1) ngưỡng trên TẬP CHỈNH
    full = np.arange(len(Y2))
    P, R, F, AP = metrics_at(sc["v2"], th, full)            # 2) đo trên TẬP ĐO, có trọng số theo tầng
    rng = np.random.default_rng(seed)
    boots = np.array([metrics_at(sc["v2"], th, boot_v2(rng)) for _ in range(CFG["n_boot"])])
    ci = lambda j: [float(np.nanpercentile(boots[:, j], 2.5)), float(np.nanpercentile(boots[:, j], 97.5))]
    # 3) recall theo tầng trùng từ vựng (TF-IDF tiêu đề) — giá trị của fine-tune lộ ra ở tầng THẤP
    tiers = {n: float(np.sum(W2[m & (Y2 == 1)] * (sc["v2"][m & (Y2 == 1)] >= th)) / np.sum(W2[m & (Y2 == 1)]))
             for n, m in TIERS.items()}
    # 4) bộ dò: tỉ lệ cặp "cùng câu chuyện, khác sự kiện" vượt ngưỡng = sẽ bị gộp nhầm (thấp hơn là tốt hơn)
    hits = sc["probe"] >= th
    pb = np.array([hits[rng.integers(0, len(hits), len(hits))].mean() for _ in range(CFG["n_boot"])])
    return dict(threshold=th, precision=P, recall=R, f1=F, f1_ci95=ci(2), precision_ci95=ci(0), recall_ci95=ci(1),
                ap=AP, ap_ci95=ci(3), recall_by_lexical_tier=tiers,
                probe_merge_rate=float(hits.mean()),
                probe_merge_rate_ci95=[float(np.percentile(pb, 2.5)), float(np.percentile(pb, 97.5))],
                probe_mean_cos=float(sc["probe"].mean()), graded=graded_metrics(sc["v4"]),
                hard=hard_metrics(sc["v4h"], th))


def compare(sa, sb, seed=0):
    # Bootstrap GHÉP CẶP: cùng một lần lấy mẫu lại cho cả hai mô hình. Lấy mẫu lại cả tập chỉnh và chọn lại ngưỡng
    # mỗi lần → CI gồm cả độ bất định của ngưỡng. Chênh = B − A.
    rng = np.random.default_rng(seed)
    low = TIERS["low"] & (Y2 == 1)
    def auc(s, y, hi, lo):
        a, b = s[y == hi], s[y == lo]
        return float((a[:, None] > b[None, :]).mean() + 0.5 * (a[:, None] == b[None, :]).mean()) if len(a) and len(b) \
            else float("nan")
    def stats(sc, i1, i2, ip, i4, ih):
        th = best_threshold(sc["v1"][i1], Y1[i1])
        _, _, f, ap = metrics_at(sc["v2"], th, i2)
        m = low[i2]
        lo = np.sum(W2[i2][m] * (sc["v2"][i2][m] >= th)) / np.sum(W2[i2][m])
        g = [auc(sc["v4"][i4], L4[i4], 3, 2), auc(sc["v4"][i4], L4[i4], 2, 1)] if len(L4) else [np.nan, np.nan]
        hp = ih[L4H[ih] == 3] if len(L4H) else ih
        rh = float((sc["v4h"][hp] >= th).mean()) if len(hp) else np.nan
        return np.array([f, ap, lo, (sc["probe"][ip] >= th).mean(), *g, rh])
    full = (np.arange(len(Y1)), np.arange(len(Y2)), np.arange(len(sa["probe"])), np.arange(len(L4)), np.arange(len(L4H)))
    point = stats(sb, *full) - stats(sa, *full)
    deltas = []
    for _ in range(CFG["n_boot"]):
        idx = (rng.integers(0, len(Y1), len(Y1)), boot_v2(rng), rng.integers(0, len(sa["probe"]), len(sa["probe"])),
               rng.integers(0, len(L4), len(L4)), rng.integers(0, len(L4H), len(L4H)))
        deltas.append(stats(sb, *idx) - stats(sa, *idx))
    deltas = np.array(deltas)
    out = {}
    for j, name in enumerate(["f1", "ap", "recall_low_tier", "probe_merge_rate", "auc_3v2", "auc_2v1", "recall_hard"]):
        out[name] = dict(delta=float(point[j]),
                         ci95=[float(np.nanpercentile(deltas[:, j], 2.5)), float(np.nanpercentile(deltas[:, j], 97.5))],
                         p_le_0=float(np.mean(deltas[:, j] <= 0)))
    return out


def show(name, m):
    t = m["recall_by_lexical_tier"]
    print(f"{name:16s} ngưỡng={m['threshold']:.3f}  P={m['precision']:.3f} R={m['recall']:.3f} "
          f"F1={m['f1']:.3f} CI95[{m['f1_ci95'][0]:.3f},{m['f1_ci95'][1]:.3f}]  AP={m['ap']:.3f}  "
          f"recall thấp/vừa/cao={t['low']:.2f}/{t['mid']:.2f}/{t['high']:.2f}  "
          f"gộp nhầm bộ dò={m['probe_merge_rate']:.2f}")
    g = m.get("graded") or {}
    if g:
        print(f"{'':16s} 4 mức: AUC 3>2 {g.get('auc_3v2', float('nan')):.3f} · 2>1 {g.get('auc_2v1', float('nan')):.3f} · "
              f"1>0 {g.get('auc_1v0', float('nan')):.3f} · Spearman {g['spearman']:.3f}")
    h = m.get("hard") or {}
    if h:
        print(f"{'':16s} khác chữ ({h['n']} cặp, {h['n_level3']} mức 3): recall {h['recall_hard']:.3f} · "
              f"gộp nhầm {h['false_merge_hard']:.3f} · AUC 3>2 {h.get('hard_auc_3v2', float('nan')):.3f}")


def show_cmp(name, c):
    print(f"{name:16s} " + "  ".join(
        f"Δ{k}={v['delta']:+.3f} [{v['ci95'][0]:+.3f},{v['ci95'][1]:+.3f}]" for k, v in c.items()))


dev_eval = PairRetrievalEvaluator(dev_pos, dev_pool, name="dev")
print(f"dev: {len(dev_eval.queries)} truy vấn trên bể {len(dev_eval.pool)} bài")

## 5. Mốc zero-shot

Số đo trước đây (cùng giao thức): bkai zero-shot F1 0,610 [0,51; 0,70], AP 0,688; TF-IDF tiêu đề F1 0,594, AP 0,587.
Nếu ô dưới ra khác nhiều → kiểm tra lại tiền xử lý. (MRR dev **không** so được với lần chạy 10/10: bể truy hồi giờ
gồm mọi bài trong khoảng dev, và dev của `v3` khác dev của `temp`.)

In [ ]:
from sentence_transformers import SentenceTransformer

base = SentenceTransformer(CFG["model_name"], device="cuda" if torch.cuda.is_available() else "cpu")
base.max_seq_length = CFG["max_seq_length"]
zero_dev = dev_eval(base)
zero_scores = pair_scores(base)
zero_audit = summarize(zero_scores)
print("dev zero-shot:", {k: round(v, 4) for k, v in zero_dev.items()})
show("zero-shot", zero_audit)
del base

## 6. Huấn luyện

- `gist` / `mnrl` học từ **cặp dương** (cả hai chiều a→b, b→a) và, với `v3`, **negative khó** của bộ ba; negative
  còn lại lấy từ các bài khác trong batch. Cặp âm của file tạm quá dễ (khác chuyên mục, cách hàng chục ngày) nên
  không dùng với hai loss này.
- `ClusterBatchSampler`: trong một batch không bài nào lặp lại và không hai dòng nào chung cụm sự kiện.
- Checkpoint tốt nhất chọn theo `eval_dev_mrr` (chỉ dev, **không** nhìn tập đo).

In [ ]:
import gc
from datasets import Dataset
from sentence_transformers import (SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses)
from sentence_transformers.sampler import DefaultBatchSampler
from sentence_transformers.training_args import BatchSamplers


class ClusterBatchSampler(DefaultBatchSampler):
    # Gán ClusterBatchSampler.keys = [tập cụm của dòng i] trước khi train. Truyền LỚP (không phải hàm) vào
    # training args để Trainer pickle được khi lưu checkpoint.
    keys: list = []

    def __init__(self, dataset, batch_size, drop_last, valid_label_columns=None, generator=None, seed=0):
        super().__init__(dataset, batch_size=batch_size, drop_last=drop_last,
                         valid_label_columns=valid_label_columns, generator=generator, seed=seed)
        assert len(self.keys) == len(dataset), "keys không khớp dataset"
        self._len = sum(1 for _ in self._batches(torch.Generator().manual_seed(seed)))

    def _batches(self, gen):
        remaining = torch.randperm(len(self.keys), generator=gen).tolist()
        while remaining:
            batch, used, rest = [], set(), []
            for i in remaining:
                if len(batch) < self.batch_size and not (self.keys[i] & used):
                    batch.append(i)
                    used |= self.keys[i]
                else:
                    rest.append(i)
            if self.drop_last and len(batch) < self.batch_size:
                return
            yield batch
            remaining = rest

    def __iter__(self):
        gen = self.generator if self.generator is not None else torch.Generator()
        gen.manual_seed(self.seed + self.epoch)
        yield from self._batches(gen)

    def __len__(self):
        return self._len


def build_dataset():
    if CFG["loss"] in ("cosent", "angle"):
        data = [{"sentence1": seg(r["a"]), "sentence2": seg(r["b"]), "score": r["score"]} for r in train_rows]
        keys = None
    else:
        data, keys = [], []
        # negative nằm cùng cụm với anchor (nối qua bài trung gian) là mâu thuẫn bắc cầu → bỏ dòng đó (v3: 2 dòng)
        with_neg = [r for r in train_rows if r["label"] == 1 and r["neg"] and cluster(r["neg"]) != cluster(r["a"])]
        use = with_neg if with_neg else [r for r in train_rows if r["label"] == 1]   # cột phải đồng nhất
        for r in use:
            for x, y in ((r["a"], r["b"]), (r["b"], r["a"])):                       # tác vụ đối xứng
                d = {"anchor": seg(x), "positive": seg(y)}
                if r["neg"]:
                    d["negative"] = seg(r["neg"])
                data.append(d)
                keys.append({cluster(i) for i in row_ids(r)})
    if CFG["smoke"]:
        pick = sorted(random.Random(0).sample(range(len(data)), min(256, len(data))))
        data = [data[i] for i in pick]
        keys = [keys[i] for i in pick] if keys is not None else None
    return Dataset.from_list(data), keys


def make_loss(model):
    if CFG["loss"] == "gist":
        guide = SentenceTransformer(CFG["model_name"], device=model.device)   # bản zero-shot, không cập nhật
        guide.max_seq_length = CFG["max_seq_length"]
        return losses.GISTEmbedLoss(model, guide, temperature=CFG["temperature"])
    if CFG["loss"] == "mnrl":
        return losses.MultipleNegativesRankingLoss(model, scale=1 / CFG["temperature"])
    if CFG["loss"] == "cosent":
        return losses.CoSENTLoss(model)
    if CFG["loss"] == "angle":
        return losses.AnglELoss(model)
    raise ValueError(CFG["loss"])


def train_one(seed):
    tag = f"bkai-{CFG['data']}-{CFG['loss']}-s{seed}"
    model = SentenceTransformer(CFG["model_name"], device="cuda" if torch.cuda.is_available() else "cpu")
    model.max_seq_length = CFG["max_seq_length"]
    ds, keys = build_dataset()
    if CFG["loss"] in ("cosent", "angle"):
        sampler = BatchSamplers.BATCH_SAMPLER
    elif CFG["group_batches"]:
        ClusterBatchSampler.keys = keys
        sampler = ClusterBatchSampler
    else:
        sampler = BatchSamplers.NO_DUPLICATES
    args = SentenceTransformerTrainingArguments(
        output_dir=str(OUT / "checkpoints" / tag), seed=seed,
        num_train_epochs=CFG["epochs"], per_device_train_batch_size=CFG["batch_size"],
        learning_rate=CFG["lr"], warmup_ratio=CFG["warmup_ratio"], weight_decay=CFG["weight_decay"],
        fp16=torch.cuda.is_available(), batch_sampler=sampler,
        eval_strategy="steps", eval_steps=CFG["eval_steps"],
        save_strategy="steps", save_steps=CFG["eval_steps"], save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="eval_dev_mrr", greater_is_better=True,
        logging_steps=10, report_to="none",
        max_steps=3 if CFG["smoke"] else -1,
    )
    trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=ds,
                                         loss=make_loss(model), evaluator=dev_eval)
    print(f"[{tag}] {len(ds)} mẫu · cột {ds.column_names} · {len(trainer.get_train_dataloader())} batch/epoch")
    trainer.train()
    dev_m = dev_eval(model)
    sc = pair_scores(model)
    audit_m = summarize(sc, seed=seed)
    cmp_m = compare(zero_scores, sc, seed=seed)
    save_dir = OUT / "models" / tag
    model.save(str(save_dir))
    (save_dir / "PREPROCESS.md").write_text(
        "Đầu vào: tiêu đề + '. ' + sapo → unicodedata.normalize('NFC') → pyvi.ViTokenizer.tokenize → "
        "model.encode(..., normalize_embeddings=True).\n", encoding="utf-8")
    log = [h for h in trainer.state.log_history]
    del trainer, model
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return dict(tag=tag, seed=seed, dev=dev_m, audit=audit_m, vs_zero_shot=cmp_m, n_train=len(ds),
                scores={k: v.round(5).tolist() for k, v in sc.items()}, log_history=log)


runs = [train_one(s) for s in CFG["seeds"]]

## 7. Kết quả

Cách đọc nhanh:
- `Δf1`, `Δap` > 0 và **CI không chứa 0** → fine-tune tốt hơn zero-shot trên tập đo (có kể độ bất định ngưỡng).
- `Δrecall_low_tier` > 0 → tăng đúng chỗ cần (cặp viết khác chữ).
- `Δrecall_hard` > 0 → nhận ra được nhiều hơn các cặp cùng sự kiện nhưng viết khác chữ (dòng "khác chữ"; cần `v4_test_hard.csv`).
- `Δauc_3v2` > 0 và CI không chứa 0 → tách "cùng sự kiện" khỏi "cùng câu chuyện" tốt hơn zero-shot (chỉ số chính của v4).
- `Δprobe_merge_rate` **< 0 là tốt**; > 0 nghĩa là mô hình gộp nhầm nhiều hơn các bài cùng câu chuyện / cùng khuôn tin.
- Dòng "4 mức" (khi dataset có `v4_test.csv`): **AUC 3>2** là chỉ số quan trọng nhất cho gom cụm — mô hình có tách được
  "cùng sự kiện" khỏi "cùng câu chuyện, khác sự kiện" không. 0,5 = không tách được.
- Nhiều seed: trung bình ± độ lệch chuẩn phải lớn hơn hẳn độ lệch thì mới coi là ổn định.

In [ ]:
show("zero-shot", zero_audit)
for r in runs:
    show(r["tag"], r["audit"])
    show_cmp("  so zero-shot", r["vs_zero_shot"])
    print(f"{'':16s} dev MRR {zero_dev['dev_mrr']:.3f} → {r['dev']['dev_mrr']:.3f} · "
          f"hit@1 {zero_dev['dev_hit@1']:.3f} → {r['dev']['dev_hit@1']:.3f}")

summary = {}
if len(runs) > 1:
    for key, get in [("f1", lambda a: a["f1"]), ("ap", lambda a: a["ap"]),
                     ("recall_low_tier", lambda a: a["recall_by_lexical_tier"]["low"]),
                     ("probe_merge_rate", lambda a: a["probe_merge_rate"]),
                     ("auc_3v2", lambda a: (a.get("graded") or {}).get("auc_3v2", float("nan"))),
                     ("spearman_4_muc", lambda a: (a.get("graded") or {}).get("spearman", float("nan"))),
                     ("recall_hard", lambda a: (a.get("hard") or {}).get("recall_hard", float("nan"))),
                     ("false_merge_hard", lambda a: (a.get("hard") or {}).get("false_merge_hard", float("nan")))]:
        v = np.array([get(r["audit"]) for r in runs])
        summary[key] = dict(mean=float(v.mean()), std=float(v.std(ddof=1)), zero_shot=float(get(zero_audit)))
        print(f"{key:18s} qua {len(runs)} seed: {v.mean():.3f} ± {v.std(ddof=1):.3f}  (zero-shot {get(zero_audit):.3f})")

results = dict(config=CFG, versions=dict(torch=torch.__version__, sentence_transformers=sentence_transformers.__version__,
                                         transformers=transformers.__version__, pyvi=version("pyvi")),
               data=dict(n_rows=len(rows), n_train_rows=len(train_rows), n_dev_pos=len(dev_pos),
                         dev_pool=len(dev_pool), dev_from=str(dev_from), cut=str(cut),
                         n_clusters=len(cl_size), largest_clusters=[n for _, n in cl_size.most_common(5)],
                         n_probe=len(probe)),
               zero_shot=dict(dev=zero_dev, audit=zero_audit,
                              scores={k: v.round(5).tolist() for k, v in zero_scores.items()}),
               runs=runs, summary=summary,
               note="Nhãn đối chiếu (audit_v1/v2) là của Claude, chưa phải người. Bộ dò lấy bằng luật, chưa kiểm nhãn.")
out_file = OUT / f"results_{CFG['data']}_{CFG['loss']}.json"
out_file.write_text(json.dumps(results, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("đã ghi", out_file)

## 8. Lấy kết quả về

- **Save Version → Save & Run All** để chạy nền; khi xong, tab *Output* có `models/<tag>/` và
  `results_<data>_<loss>.json`. Tải JSON về, đặt vào `data/temp/kaggle_runs/<ngày>_bkai-<loss>-<data>/`.
- Muốn chạy cả `v3` và `temp`: sửa `CFG["data"]` rồi chạy lại (hoặc tạo 2 version notebook).
- Ô dưới cùng tự xoá `checkpoints/` và nén mô hình tốt nhất (theo dev MRR) thành `best_model_<tag>.zip`.
- **Chạy bằng Save Version → Save & Run All** để file được giữ lại; chạy tương tác thì file mất khi phiên tắt.
- (Tuỳ chọn) đẩy mô hình lên Hugging Face **riêng tư**: thêm `HF_TOKEN` vào *Add-ons → Secrets*, rồi
  `from kaggle_secrets import UserSecretsClient; token = UserSecretsClient().get_secret("HF_TOKEN")` và
  `SentenceTransformer(path).push_to_hub("<user>/<tên>", private=True, token=token)`. Không dán token vào ô code.

In [ ]:
# Dọn đầu ra: xoá checkpoint trung gian (nặng, có cả trạng thái optimizer) và nén mô hình tốt nhất THEO DEV
# (không chọn theo tập đo) thành một file zip để tải về.
import shutil
shutil.rmtree(OUT / "checkpoints", ignore_errors=True)
best = max(runs, key=lambda r: r["dev"]["dev_mrr"])
zip_path = shutil.make_archive(str(OUT / f"best_model_{best['tag']}"), "zip", root_dir=OUT / "models", base_dir=best["tag"])
print(f"mô hình chọn theo dev MRR: {best['tag']} (dev MRR {best['dev']['dev_mrr']:.3f})")
for f in sorted(OUT.iterdir()):
    size = sum(x.stat().st_size for x in f.rglob("*")) if f.is_dir() else f.stat().st_size
    print(f"  {f.name:45s} {size / 1e6:8.1f} MB")
print("Tải về: thanh bên phải → Output (/kaggle/working) → ⋮ → Download;",
      "hoặc sau khi Save Version chạy xong: tab Output của version.")